[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/en_clase/S11_03_deteccion_segmentacion_en_clase.ipynb)

# S11 · Práctica 3 — Detección de objetos y segmentación · versión en clase

**Tiempo estimado:** 35 min en clase · **Sprint 11** (UD 4.6)

📖 *¿El código se hace largo? Léelo [explicado por escrito](https://github.com/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/codigo_explicado/S11_03_deteccion_segmentacion.md): recorrido, glosario y qué hace cada celda.* <!--codigo-explicado-->

**Contexto TurisData.** Clasificar una foto dice *qué es*; pero al cliente le interesan preguntas como
*"¿cuántas sombrillas hay en la playa del resort?"* (detección) o *"¿qué parte de la imagen es piscina?"*
(segmentación). En esta práctica verás los conceptos, las métricas (IoU, NMS, mAP) con números pequeños y un
ejemplo completo **sin internet**, con imágenes aéreas sintéticas de una playa y con OpenCV.

**Al terminar sabrás:**
1. Distinguir clasificación, detección y segmentación (semántica frente a instancias).
2. Calcular la **IoU**, aplicar **NMS** y entender **precisión, recall, AP y mAP** con ejemplos numéricos.
3. Construir un detector clásico con OpenCV (color + contornos, y *template matching*) y evaluarlo.
4. Medir una segmentación (IoU/Dice de máscaras) y entrenar una mini red encoder-decoder.
5. Saber cuándo pasar a un detector moderno (YOLO) y qué cambia.

**Dos ramas.** Todo el notebook se ejecuta **offline**. Al final hay una celda **opcional de Colab con YOLO
(ultralytics)** marcada como *requiere internet* y **no verificada**.

> 🏫 **Versión «en clase»** · ⏱ ≈ 35 min <!--reparto-->
>
> - Es **la parte de este cuaderno que se hace en el aula**, con el docente: ejercicios **1, 2**.
> - Lo demás (ejercicios 🏠 y mini-reto) está en el [cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/para_casa/S11_03_deteccion_segmentacion.ipynb), para hacer en casa.


In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os, sys, subprocess
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw

try:
    import cv2
except ImportError:      # en Colab y en el taller ya viene instalado; esta celda solo actúa si faltara
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"])
    import cv2

SEMILLA = 42
print("OpenCV", cv2.__version__)

## 1. Tres tareas con imágenes

| Tarea | Pregunta | Salida | Ejemplo TurisData |
|---|---|---|---|
| **Clasificación** | ¿Qué hay en la foto? | una etiqueta | "esta foto es de una piscina" |
| **Detección** | ¿Qué hay y **dónde**? | cajas (x1, y1, x2, y2) + clase + confianza | contar sombrillas en la playa |
| **Segmentación semántica** | ¿A qué clase pertenece **cada píxel**? | una máscara | superficie de piscina en la foto |
| **Segmentación de instancias** | Lo mismo, separando cada objeto | una máscara por objeto | cada tumbona por separado |

Los detectores modernos (YOLO, Faster R-CNN, DETR) son redes convolucionales que predicen muchas cajas candidatas de una vez;
después se limpian con **NMS** y se evalúan con **mAP**. Empezamos por esas piezas.

## 2. Cajas y la IoU (*Intersection over Union*)

**Idea.** Una caja se escribe `[x1, y1, x2, y2]` (esquina superior izquierda y esquina inferior derecha, en píxeles).
Para saber si una caja predicha "acierta" con la real, se mide cuánto se solapan:

`IoU = área de la intersección / área de la unión`

0 = no se tocan; 1 = idénticas. Por convención, una detección es correcta (verdadero positivo) si su IoU con una caja real es **≥ 0,5**.

**Ejemplo:** cajas `[0,0,10,10]` y `[5,5,15,15]`: intersección 5×5 = 25; unión 100 + 100 − 25 = 175; IoU = 25/175 ≈ 0,143.

**Ejercicio 1 · 🏫 en clase.** Escribe `iou(a, b)` que reciba dos cajas `[x1, y1, x2, y2]` y devuelva su IoU (un número entre 0 y 1).
Cuidado con el caso en que no se solapan (la intersección vale 0, no negativa).

In [ ]:
# TODO: ancho y alto de la intersección con max(0, ...); unión = área_a + área_b - intersección
...

assert abs(iou([0, 0, 10, 10], [5, 5, 15, 15]) - 25 / 175) < 1e-9, "IoU esperada 25/175 ≈ 0.143"
assert iou([0, 0, 10, 10], [0, 0, 10, 10]) == 1.0, "Cajas idénticas: IoU = 1"
assert iou([0, 0, 10, 10], [20, 20, 30, 30]) == 0.0, "Cajas separadas: IoU = 0 (¿has acotado con max(0, ...)?)"
assert abs(iou([0, 0, 10, 10], [0, 0, 10, 5]) - 0.5) < 1e-9, "Una caja dentro de otra: IoU = área pequeña / área grande"

## 3. NMS (*Non-Maximum Suppression*): quedarse con una caja por objeto

**Idea.** Un detector suele proponer **varias cajas casi iguales** para el mismo objeto. NMS las limpia:
1. Ordena las cajas por confianza (de mayor a menor).
2. Se queda con la primera.
3. **Descarta** las demás que se solapan con ella más de un umbral (por ejemplo IoU > 0,5).
4. Repite con las que quedan.

**Ejemplo:** cajas A (confianza 0,9), B (0,8, casi igual a A) y C (0,7, lejos). NMS conserva A y C: B era un duplicado de A.

**Ejercicio 2 · 🏫 en clase.** Escribe `nms(cajas, puntuaciones, umbral_iou=0.5)` que devuelva la **lista de índices** de las cajas que se
conservan, en orden de confianza decreciente. Usa tu función `iou`.

In [ ]:
# TODO: Ordena índices por puntuación; mientras queden, guarda el primero y filtra los que tienen iou > umbral con él
...

_cajas = [[10, 10, 50, 50], [12, 12, 52, 52], [100, 100, 140, 140], [11, 9, 49, 51]]
_punt = [0.9, 0.8, 0.7, 0.6]
assert nms(_cajas, _punt) == [0, 2], "Debe quedar la caja 0 (mejor) y la 2 (lejana); la 1 y la 3 son duplicados de la 0"
assert nms(_cajas, _punt, umbral_iou=0.99) == [0, 1, 2, 3], "Con un umbral muy alto no se descarta nada"
assert nms([[0, 0, 5, 5]], [0.3]) == [0], "Con una sola caja se conserva"

---

## 🏁 Fin de la parte de clase

- Lo que queda (los ejercicios 🏠 y el mini-reto) está en el **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/para_casa/S11_03_deteccion_segmentacion.ipynb)**.
- Ahí también tienes, en cada ejercicio de casa, su celda plegada «🆘 Solución de referencia» por si te quedas atascado.
